In [0]:
import pandas as pd
from data import generate_messy_data

## Cleaning & Transforming 

_INGESTION: Standardize Header_

In [0]:
df = generate_messy_data()

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

_Duplicates_

In [0]:
#no parameterization, risky for large dataset
df= df.drop_duplicates() 
'''
#uses a singlular parameterization, to check for duplicates using a specific column and keeps the first occurence
df= df.drop_duplicates(subset=['user_id'])
#using multiple parameters to check for duplicates using multiple columns is more efficient and unique
df= df.drop_duplicates(subset=['user_id', "sale_date"])
#you can also use a lambda function to check for duplicates based on a specific condition
df= df.drop_duplicates(subset=['user_id', "sale_date"], keep='last') #keeps the last occurence
df= df.drop_duplicates(subset=['user_id', "sale_date"], keep='first') #keeps all first occurences
df= df.drop_duplicates(subset=['user_id', "sale_date"], keep=False) #deletes any and all occurences
'''


In [0]:
df['user_id']= df['user_id'].interpolate(method='linear')

In [0]:
df['email_addr'] = df['email_addr'].str.replace(r'#', '@', regex=True) #to replace a specific character with another character



_Handling Nulls_

In [0]:
df

In [0]:
if (df['region_name'] == 'unknown').any():
    print(df['region_name'].unique())

In [0]:
df['sale_date'] = df['sale_date'].astype('datetime64[ns]')


In [0]:
df['sale_date'] = df['sale_date'].interpolate(method='linear')

In [0]:
df['region_name'] = df['region_name'].str.lower()
df['region_name']=df['region_name'].astype('category')
df['sale_date'] = pd.to_datetime(df['sale_date'], format='%b %dth, %Y', errors='coerce')
# If sale_date is missing, subtract 2 days from the shipping date
df['sale_date'] = df['sale_date'].fillna(df['sale_date'] - pd.Timedelta(days=2))
df['base_price'] = df['base_price'].clip(lower=0, upper=500)


In [0]:
# Force all ratings to be between 1 and 5
df['user_rating'] = df['user_rating'].clip(upper=5)


In [0]:
df= df.fillna(0)    #to replace nulls with a specified value
'''
df= df.dropna()     #to remove rows that contain null values
df= df.dropna(subset=['user_id'])   #to remove rows that contain null values in a specific column
df= df.dropna(subset=['user_id', "email_addr"])   #to remove columns that contain null values in multiple columns
df= df.ffill()      #to replace nulls with the previous value
df= df.bfill()      #to replace nulls with the next value
df= df.fillna(method='ffill') #to replace nulls with the previous value
df= df.fillna(method='bfill') #to replace nulls with the next value
df= df.dropna(axis=1) #removes all columns from your DataFrame that contain any null (NaN) values. This is equivalent to df.dropna(how='any', axis=1).
df= df.dropna(axis=0) #removes all rows from your DataFrame that contain any null (NaN) values. This is equivalent to df.dropna(how='any', axis=0).

df= df.dropna(how='any') #removes all rows from your DataFrame that contain any null (NaN) values. This is equivalent to df.dropna(how='any', axis=0).
df= df.dropna(how='any', axis=1) #removes all columns from your DataFrame that contain any null (NaN) values. This is equivalent to df.dropna(how='any', axis=1).
df= df.dropna(how='any', axis=0) #removes all rows from your DataFrame that contain any null (NaN) values. This is equivalent to df.dropna(how='any', axis=0).

df= df.dropna(how='all') #removes all rows from your DataFrame that contain all null (NaN) values. This is equivalent to df.dropna(how='all', axis=0).
df= df.dropna(how='all', axis=1) #removes all columns from your DataFrame that contain all null (NaN) values. This is equivalent to df.dropna(how='all', axis=1).
df= df.dropna(how='all', axis=0) #removes all rows from your DataFrame that contain all null (NaN) values. This is equivalent to df.dropna(how='all', axis=0).

df= df.dropna(thresh=1) #removes all rows from your DataFrame that contain less than 1 non-null (NaN) values. This is equivalent to df.dropna(thresh=1, axis=0).
df= df.dropna(thresh=2) #removes all rows from your DataFrame that contain less than 2 non-null (NaN) values. This is equivalent to df.dropna(thresh=2, axis=0).
'''


In [0]:
df = df.drop('normal_rating', axis=1)

In [0]:
print(df.head())

In [0]:
df

In [0]:
# Convert Pandas to Spark
spark_df = spark.createDataFrame(df)

# Now save it to the database
spark_df.write.mode("overwrite").saveAsTable("transform.sql_db.python_table")

In [0]:
%sql
select * from transform.sql_db.python_table
